# Gerador de Cartas de Magic: The Gathering

Projeto da disciplina de IA e Bibliotecas de Dados para Python.

**Como usar:** no menu acima, clique em **Ambiente de execução > Executar tudo** (ou `Ctrl+F9`). As duas primeiras células preparam o ambiente (clonam o repositório e instalam as dependências); a terceira mostra o formulário para gerar a carta.

O código completo do pipeline de geração está em [`src/pipeline/generator.py`](../src/pipeline/generator.py) no repositório — este notebook só importa e chama a função `gerar_carta`, para que a mesma lógica possa também ser usada por outras interfaces no futuro (ex: um app web).

## 1. Preparar o ambiente

Clona o repositório (se ainda não estiver presente) e instala as dependências.

In [ ]:
import os

REPO_URL = "https://github.com/funkymonks21/MTGCardCreator-IFRS.git"
REPO_DIR = "MTGCardCreator-IFRS"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
else:
    # a pasta ja existe de uma execucao anterior nesta sessao do Colab --
    # puxa os commits novos em vez de ignorar
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "src")

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Ambiente pronto.")

## 2. Carregar o pipeline

Importa a função `gerar_carta`. Dois conjuntos de modelos são carregados automaticamente, sob demanda, direto do repositório, então ninguém precisa treinar nada aqui:

- **texto** -- cadeia de Markov por cor (`models/modelo_markov.pkl`, ver `src/pipeline/texto_markov.py`);
- **custo de mana e poder/resistência** -- regressões scikit-learn (`models/modelos_estruturais.pkl`, ver `src/pipeline/estrutura.py`).

In [ ]:
from pipeline import gerar_carta, COLOR_PIE, PESO_COR_PADRAO

print("Pipeline carregado. Cores disponiveis:", list(COLOR_PIE))
print("Peso padrao da cor na geracao (0=caos total, 1=so vocabulario da cor):", PESO_COR_PADRAO)

### Teste rápido (sem usar os campos abaixo)

A geração agora é uma cadeia de Markov (bigramas/trigramas) treinada por cor, com peso ajustável entre o vocabulário típico da cor e um modelo geral -- **sem** filtro de color pie, de propósito (ver decisão de design em `src/pipeline/generator.py`). Rodar a mesma carta várias vezes deve dar textos diferentes.

In [ ]:
exemplos = [
    ("Murder", "Azul", 1.0),       # semente "destroy" via dicionario/WordNet, so vocabulario azul
    ("Murder", "Azul", 0.3),       # mesma semente, mas mais peso pro modelo geral (mais caos)
    ("Murder", "Preto", 0.7),
    ("jasdhahs12123", "Verde", 0.7),  # nome sem mecanica reconhecivel -> sem semente
]

for nome_exemplo, cor_exemplo, peso_exemplo in exemplos:
    c = gerar_carta(nome_exemplo, cor_exemplo, peso_cor=peso_exemplo)
    print(f"{nome_exemplo!r} + {cor_exemplo} (peso_cor={peso_exemplo}):")
    print(f"  semente: {c.semente_usada or '(nenhuma)'}")
    print(f"  custo: {c.custo_mana} | P/T: {c.poder}/{c.resistencia}")
    print(f"  texto: {c.texto_regras}\n")

## 3. Gerar sua carta

Preencha o nome, escolha a cor e ajuste o quanto a cor deve pesar na geração, depois clique em **Gerar carta**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

campo_nome = widgets.Text(
    description="Nome:",
    placeholder="ex: Murder, Lightning Bolt, jasdhahs12123...",
    style={"description_width": "initial"},
)
campo_cor = widgets.Dropdown(
    options=list(COLOR_PIE.keys()),
    description="Cor:",
    style={"description_width": "initial"},
)
campo_peso_cor = widgets.FloatSlider(
    value=PESO_COR_PADRAO, min=0.0, max=1.0, step=0.1,
    description="Peso da cor:",
    style={"description_width": "initial"},
    readout_format=".1f",
)
botao_gerar = widgets.Button(description="Gerar carta", button_style="success")
saida = widgets.Output()


def ao_clicar(_botao):
    with saida:
        clear_output(wait=True)
        carta = gerar_carta(campo_nome.value, campo_cor.value, peso_cor=campo_peso_cor.value)
        display(carta.imagem_final)
        print(f"Nome: {carta.nome}")
        print(f"Custo de mana: {carta.custo_mana}")
        print(f"Poder/Resistência: {carta.poder}/{carta.resistencia}")
        print(f"Tipo: {carta.tipo}")
        print(f"Texto: {carta.texto_regras}")
        print()
        print(f"[debug] peso da cor usado: {carta.peso_cor}")
        print(f"[debug] palavra-semente: {carta.semente_usada or '(nenhuma -- cadeia comecou do zero)'}")


botao_gerar.on_click(ao_clicar)
display(campo_nome, campo_cor, campo_peso_cor, botao_gerar, saida)